In [ ]:
!pip install transformer_lens

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 7.1 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=9c69ae365eff5110c8cd1904ddf3b6e2fd2d8ffcefe177e0e37159e8172e524e
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


In [ ]:
import torch
import numpy as np

def generate_with_steering(
    model,
    prompt: str,
    target_layers: list,
    layer_dirs: dict,
    alpha: float = 0.0,
    position: int = -1,
    top_k: int = 5
):
    """
    Applies a Steering Vector (alpha * v) to specified accepted layers in real-time.

    Args:
        model: HookedTransformer instance
        prompt: Input string (e.g., "2 + 1 =")
        target_layers: List of ACCEPTED layers, e.g. [7, 8, 9, 10, 11]
        layer_dirs: Calibrated probe vectors dictionary
        alpha: Steering intensity (+ = increase sum value, - = decrease)
        position: Target token index (-1 = last prompt token)
        top_k: Number of top logits to display
    """
    tokens = model.to_tokens(prompt)

    # 1. Define hook generator for selected layers
    def make_steering_hook(layer_idx):
        # Convert probe direction array to GPU tensor matching model device
        v = torch.tensor(
            layer_dirs[layer_idx],
            dtype=torch.float32,
            device=model.cfg.device
        )
        def hook_fn(activation, hook):
            # Inject steering vector: activation + (alpha * v)
            activation[:, position, :] = activation[:, position, :] + alpha * v
            return activation
        return hook_fn

    hooks = [
        (f"blocks.{l}.hook_resid_post", make_steering_hook(l))
        for l in target_layers
    ]

    # 2. Forward pass with steering hooks
    steered_logits = model.run_with_hooks(tokens, fwd_hooks=hooks)[0, -1, :]

    # 3. Extract top-K predicted tokens and probabilities
    probs = torch.softmax(steered_logits, dim=-1)
    topk_vals, topk_indices = torch.topk(probs, k=top_k)

    predictions = []
    for val, idx in zip(topk_vals, topk_indices):
        tok_str = model.to_string(torch.tensor([idx.item()]))
        predictions.append((tok_str.strip(), float(val)))

    top_token = predictions[0][0]
    return top_token, predictions


# =============================================================================
# LIVE DEMO: Closed-Loop Steering Test on Accepted Layers (7 to 11)
# =============================================================================

ACCEPTED_LAYERS = [7, 8, 9, 10, 11]
test_prompt = "2 + 1 ="

print("=" * 65)
print(f"LIVE STEERING EXPERIMENT ON PROMPT: '{test_prompt}'")
print(f"Accepted Causal Layers: {ACCEPTED_LAYERS}")
print("=" * 65)

# Test 1: Natural Model Behavior (Alpha = 0.0)
top_tok_nat, probs_nat = generate_with_steering(
    model, test_prompt, ACCEPTED_LAYERS, layer_dirs, alpha=0.0
)
print(f"\n[1] Natural Output (Alpha = 0.0): -> Predicts '{top_tok_nat}'")
for tok, p in probs_nat:
    print(f"    Token: '{tok:<3}' | Prob: {p*100:.2f}%")

# Test 2: Positive Steering (Alpha = +8.0) -> Push towards higher sum
top_tok_pos, probs_pos = generate_with_steering(
    model, test_prompt, ACCEPTED_LAYERS, layer_dirs, alpha=8.0
)
print(f"\n[2] Positive Steering (Alpha = +8.0) -> Push Higher Value:")
print(f"    New Predicted Token: '{top_tok_pos}'")
for tok, p in probs_pos:
    print(f"    Token: '{tok:<3}' | Prob: {p*100:.2f}%")

# Test 3: Negative Steering (Alpha = -8.0) -> Push towards lower sum
top_tok_neg, probs_neg = generate_with_steering(
    model, test_prompt, ACCEPTED_LAYERS, layer_dirs, alpha=-8.0
)
print(f"\n[3] Negative Steering (Alpha = -8.0) -> Push Lower Value:")
print(f"    New Predicted Token: '{top_tok_neg}'")
for tok, p in probs_neg:
    print(f"    Token: '{tok:<3}' | Prob: {p*100:.2f}%")

In [ ]:
"""
Phase 2 -- Statistically Rigorous Open Discovery Protocol.
Fixes:
- SyntaxError in Environment.get_next_prompt_and_operand (removed double parenthesis)
- Corrected F-test Degrees of Freedom using matrix rank
- Enforced True Granger Causality
- Single-pass trajectory collection
"""

import numpy as np
import torch
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from transformer_lens import HookedTransformer

# Hyperparameters
ALPHA = 0.05
LOGIT_DIFF_THRESHOLD = 0.5
BUDGET = 24
N_STEPS = 200
LAG = 3
K_PCA = 5
N_FLIP_TRIALS = 20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.set_grad_enabled(False)

print(f"[setup] Using device: {DEVICE}")
model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
model.eval()
N_LAYERS = model.cfg.n_layers
N_HEADS = model.cfg.n_heads

def build_lagged(matrix: np.ndarray, lag: int) -> np.ndarray:
    T = matrix.shape[0]
    cols = [matrix[lag - k: T - k] for k in range(1, lag + 1)]
    return np.concatenate(cols, axis=1)

def ols_rss(y: np.ndarray, X: np.ndarray) -> float:
    X_design = np.concatenate([np.ones((X.shape[0], 1)), X], axis=1)
    beta, _, _, _ = np.linalg.lstsq(X_design, y, rcond=None)
    resid = y - X_design @ beta
    return float(resid @ resid)

def benjamini_hochberg(p_values: list, alpha: float = ALPHA) -> list:
    m = len(p_values)
    order = np.argsort(p_values)
    sorted_p = np.array(p_values)[order]
    thresholds = (np.arange(1, m + 1) / m) * alpha
    passed = sorted_p <= thresholds
    cutoff_rank = np.max(np.where(passed)) if passed.any() else -1
    reject = np.zeros(m, dtype=bool)
    if cutoff_rank >= 0:
        reject[order[: cutoff_rank + 1]] = True
    return reject.tolist()

# =============================================================================
# WorldModel & Evidence Infrastructure
# =============================================================================

class NodeStatus(Enum):
    CANDIDATE = "candidate"
    ACCEPTED = "accepted"
    REJECTED = "rejected"
    UNRESOLVED = "unresolved"

@dataclass
class Node:
    signal_id: str
    status: NodeStatus
    endogeneity_p_value: Optional[float] = None
    best_handle_id: Optional[str] = None
    delta_logit: Optional[float] = None
    flip_rate: Optional[float] = None
    evidence: dict = field(default_factory=dict)

class WorldModel:
    def __init__(self):
        self.nodes: Dict[str, Node] = {}

    def add_candidate(self, signal_id: str, p_value: float):
        if signal_id not in self.nodes:
            self.nodes[signal_id] = Node(signal_id=signal_id, status=NodeStatus.CANDIDATE,
                                          endogeneity_p_value=p_value)

    def accept(self, signal_id: str, handle_id: str, delta_logit: float, flip_rate: float, evidence: Optional[dict] = None):
        if signal_id not in self.nodes:
            self.add_candidate(signal_id, 0.0)
        node = self.nodes[signal_id]
        node.status = NodeStatus.ACCEPTED
        node.best_handle_id = handle_id
        node.delta_logit = delta_logit
        node.flip_rate = flip_rate
        if evidence:
            node.evidence.update(evidence)

    def reject(self, signal_id: str, handle_id: Optional[str] = None, delta_logit: Optional[float] = None, flip_rate: Optional[float] = None):
        if signal_id not in self.nodes:
            self.add_candidate(signal_id, 1.0)
        node = self.nodes[signal_id]
        node.status = NodeStatus.REJECTED
        if handle_id:
            node.best_handle_id = handle_id
            node.delta_logit = delta_logit
            node.flip_rate = flip_rate

    def self_model_summary(self) -> str:
        lines = []
        by_status: Dict[NodeStatus, List[str]] = {}
        for sid, node in self.nodes.items():
            by_status.setdefault(node.status, []).append(sid)
        for status in NodeStatus:
            ids = by_status.get(status, [])
            if ids:
                lines.append(f"  {status.value:<12}: {len(ids)} signal(s) -- {ids}")
        return "\n".join(lines) if lines else "  (empty)"

class NodeKind(Enum):
    MECHANISM = "mechanism"
    DECOY = "decoy"
    UNKNOWN = "unknown"

@dataclass
class SelfGraphNode:
    node_id: str
    kind: NodeKind = NodeKind.UNKNOWN
    kind_confidence: float = 0.0
    evidence: dict = field(default_factory=dict)

class KnowledgeUnitView:
    def __init__(self, kind_node: SelfGraphNode, world_model: WorldModel):
        self._kind_node = kind_node
        self._wm = world_model

    @property
    def node_id(self) -> str: return self._kind_node.node_id
    @property
    def kind(self) -> NodeKind: return self._kind_node.kind
    @property
    def status(self) -> Optional[NodeStatus]:
        node = self._wm.nodes.get(self.node_id)
        return node.status if node else None
    @property
    def delta_logit(self) -> Optional[float]:
        node = self._wm.nodes.get(self.node_id)
        return node.delta_logit if node else None
    @property
    def flip_rate(self) -> Optional[float]:
        node = self._wm.nodes.get(self.node_id)
        return node.flip_rate if node else None

    def __repr__(self):
        d_val = f"{self.delta_logit:+.3f}" if self.delta_logit is not None else "N/A"
        f_val = f"{self.flip_rate:.2f}" if self.flip_rate is not None else "N/A"
        return (f"KnowledgeUnitView({self.node_id!r}, kind={self.kind.value}, "
                f"status={self.status.value if self.status else None}, "
                f"delta_logit={d_val}, flip_rate={f_val})")

class EvidenceManager:
    def __init__(self, world_model: WorldModel, self_graph_nodes: Dict[str, SelfGraphNode]):
        self.wm = world_model
        self.sg_nodes = self_graph_nodes

    def _ensure_node(self, node_id: str) -> SelfGraphNode:
        if node_id not in self.sg_nodes:
            self.sg_nodes[node_id] = SelfGraphNode(node_id=node_id)
        return self.sg_nodes[node_id]

    def record_endogeneity(self, node_id: str, p_value: float, significant: bool) -> None:
        node = self._ensure_node(node_id)
        node.evidence["endogeneity_p_value"] = p_value
        if significant:
            self.wm.add_candidate(node_id, p_value)

    def record_intervention(self, node_id: str, handle_id: str, delta_logit: float, flip_rate: float) -> None:
        node = self._ensure_node(node_id)
        if delta_logit >= LOGIT_DIFF_THRESHOLD or flip_rate >= 0.3:
            self.wm.accept(node_id, handle_id, delta_logit, flip_rate)
            node.kind = NodeKind.MECHANISM
            node.kind_confidence = min(1.0, max(flip_rate, delta_logit / 5.0))
        else:
            self.wm.reject(node_id, handle_id, delta_logit, flip_rate)
            node.kind = NodeKind.DECOY
            node.kind_confidence = 0.5

# =============================================================================
# Environment & Discovery Setup
# =============================================================================

class Environment:
    N_STUCK_WARN = 8

    def __init__(self, model, operand_range=(1, 4), start_range=(1, 4), seed=0):
        self.model = model
        self.operand_range = operand_range
        self.start_range = start_range
        self.seed = seed
        self.rng = np.random.default_rng(seed)
        self.T = None
        self.n_anomalies = 0
        self.n_steps_run = 0
        self._stuck_count = 0
        self._last_T = None

    def reset(self):
        self.rng = np.random.default_rng(self.seed)
        self.T = int(self.rng.integers(self.start_range[0], self.start_range[1] + 1))
        self.n_anomalies = 0
        self.n_steps_run = 0
        self._stuck_count = 0
        self._last_T = None

    def get_next_prompt_and_operand(self):
        n = int(self.rng.integers(self.operand_range[0], self.operand_range[1] + 1))
        prompt = f"{self.T} + {n} ="
        return prompt, n

    def update_state(self, next_token_str: str, n: int):
        try:
            T_next = int(next_token_str.strip())
            valid = 0 <= T_next <= 9
        except ValueError:
            valid = False

        if not valid:
            self.n_anomalies += 1
            T_next_cont = (self.T + n) % 10
        else:
            T_next_cont = T_next

        if self._last_T is not None and T_next_cont == self._last_T:
            self._stuck_count += 1
            if self._stuck_count >= self.N_STUCK_WARN:
                print(f"  [WARNING] Environment T stuck at {T_next_cont} for {self._stuck_count} steps.")
        else:
            self._stuck_count = 0

        self._last_T = T_next_cont
        self.T = T_next_cont
        self.n_steps_run += 1

def action_onehot_from_digit(digit: int, n_categories: int = 10) -> list:
    row = [0.0] * n_categories
    row[digit] = 1.0
    return row

def calibrated_addition_pairs(operand_max=9, seed=7):
    candidates = [(a, b) for a in range(0, 10) for b in range(0, 10) if a + b <= operand_max]
    valid = []
    for a, b in candidates:
        ans_ids = model.to_tokens(f" {a + b}", prepend_bos=False)
        if ans_ids.shape[-1] == 1: valid.append((a, b))
    np.random.default_rng(seed).shuffle(valid)
    return valid

def calibrate_all_directions(calibration_pairs):
    layer_acts = {l: [] for l in range(N_LAYERS)}
    head_acts = {(l, h): [] for l in range(N_LAYERS) for h in range(N_HEADS)}
    sums = []
    for a, b in calibration_pairs:
        tokens = model.to_tokens(f"{a} + {b} =")
        _, cache = model.run_with_cache(tokens)
        for l in range(N_LAYERS):
            layer_acts[l].append(cache[f"blocks.{l}.hook_resid_post"][0, -1, :].cpu().numpy())
            z = cache[f"blocks.{l}.attn.hook_z"][0, -1, :, :].cpu().numpy()
            for h in range(N_HEADS): head_acts[(l, h)].append(z[h])
        sums.append(a + b)
        del cache
    sums = np.array(sums)

    layer_dirs, head_dirs = {}, {}
    for l in range(N_LAYERS):
        X = StandardScaler().fit_transform(np.stack(layer_acts[l]))
        ridge = Ridge(alpha=10.0).fit(X, sums)
        layer_dirs[l] = ridge.coef_ / (np.linalg.norm(ridge.coef_) + 1e-8)
    for key, acts in head_acts.items():
        X = StandardScaler().fit_transform(np.stack(acts))
        ridge = Ridge(alpha=10.0).fit(X, sums)
        head_dirs[key] = ridge.coef_ / (np.linalg.norm(ridge.coef_) + 1e-8)
    return layer_dirs, head_dirs

LAYER_IDS = [f"layer{l}" for l in range(N_LAYERS)]
HEAD_IDS = [f"head{l}_{h}" for l in range(N_LAYERS) for h in range(N_HEADS)]
CANDIDATE_IDS = LAYER_IDS + HEAD_IDS

print("[setup] calibrating 156 probe directions...")
layer_dirs, head_dirs = calibrate_all_directions(calibrated_addition_pairs())

# Optimized Single-Pass Trajectory Collection
def collect_trajectory(env, n_steps):
    rows, actions = [], []
    env.reset()
    for _ in range(n_steps):
        prompt, n = env.get_next_prompt_and_operand()
        tokens = model.to_tokens(prompt)
        logits, cache = model.run_with_cache(tokens)

        next_token_id = int(logits[0, -1, :].argmax().item())
        next_token_str = model.to_string(torch.tensor([next_token_id]))
        env.update_state(next_token_str, n)

        row = {}
        for l in range(N_LAYERS):
            act = cache[f"blocks.{l}.hook_resid_post"][0, -1, :].cpu().numpy()
            row[f"layer{l}"] = float(np.dot(act, layer_dirs[l]))
        for l in range(N_LAYERS):
            z = cache[f"blocks.{l}.attn.hook_z"][0, -1, :, :].cpu().numpy()
            for h in range(N_HEADS):
                row[f"head{l}_{h}"] = float(np.dot(z[h], head_dirs[(l, h)]))
        rows.append(row)
        actions.append(action_onehot_from_digit(env.T))
        del cache
    matrix = np.array([[row[cid] for cid in CANDIDATE_IDS] for row in rows])
    action_onehot = np.array(actions)
    return matrix, action_onehot, env.n_anomalies

# Corrected Granger F-Test with Exact Matrix Rank Degrees of Freedom
def conditional_granger_f_test_pca(target_col, other_matrix, action_onehot, lag=LAG, k_pca=K_PCA):
    T = target_col.shape[0]
    y = target_col[lag:]
    n = y.shape[0]
    self_lags = build_lagged(target_col.reshape(-1, 1), lag)

    if other_matrix.shape[1] > 0:
        k_eff = min(k_pca, other_matrix.shape[1])
        centered = other_matrix - other_matrix.mean(axis=0)
        _, _, Vt = np.linalg.svd(centered, full_matrices=False)
        components = centered @ Vt[:k_eff].T
        other_lags = build_lagged(components, lag)
    else:
        other_lags = np.zeros((n, 0))

    # Drop column 0 of action_onehot to avoid dummy variable trap with intercept
    action_onehot_reduced = action_onehot[:, 1:]
    action_past_lags = build_lagged(action_onehot_reduced, lag)

    restricted_X = np.concatenate([self_lags, other_lags], axis=1)
    unrestricted_X = np.concatenate([restricted_X, action_past_lags], axis=1)

    # Design matrices with global intercept
    X_r_design = np.concatenate([np.ones((restricted_X.shape[0], 1)), restricted_X], axis=1)
    X_u_design = np.concatenate([np.ones((unrestricted_X.shape[0], 1)), unrestricted_X], axis=1)

    # Exact matrix rank for correct degrees of freedom calculation
    k_r = np.linalg.matrix_rank(X_r_design)
    k_u = np.linalg.matrix_rank(X_u_design)
    q = k_u - k_r
    df2 = n - k_u

    rss_r = ols_rss(y, restricted_X)
    rss_u = ols_rss(y, unrestricted_X)

    if df2 <= 0 or rss_u <= 0 or q <= 0: return float("nan"), 1.0, df2

    f_stat = max(((rss_r - rss_u) / q) / (rss_u / df2), 0.0)
    p_value = stats.f.sf(f_stat, q, df2)
    return f_stat, p_value, df2

def test_endogeneity_batch(matrix, action_onehot, candidate_ids, lag=LAG, k_pca=K_PCA, alpha=ALPHA):
    p_values, df2s = [], []
    for i in range(matrix.shape[1]):
        target = matrix[:, i]
        others = np.delete(matrix, i, axis=1)
        _, p, df2 = conditional_granger_f_test_pca(target, others, action_onehot, lag=lag, k_pca=k_pca)
        p_values.append(p)
        df2s.append(df2)
    rejected = benjamini_hochberg(p_values, alpha=alpha)

    n_sig = sum(rejected)
    print(f"[Stage 1] True Granger Causality Screening Results:")
    print(f"  Passed (significant): {n_sig}/{len(candidate_ids)} ({n_sig/len(candidate_ids)*100:.1f}%)")
    print(f"  Exact df2 across tests: min={min(df2s)}, mean={np.mean(df2s):.1f}")

    return {
        candidate_ids[i]: {"p_value": p_values[i], "significant": rejected[i], "df2": df2s[i]}
        for i in range(len(candidate_ids))
    }

def plan_round_stratified(endog_results, budget=BUDGET):
    sig_map = {cid: r for cid, r in endog_results.items() if r["significant"]}

    layer_cids = [f"layer{l}" for l in range(N_LAYERS) if f"layer{l}" in sig_map]
    layer_cids.sort(key=lambda cid: sig_map[cid]["p_value"])

    top_heads_per_layer = []
    for l in range(N_LAYERS):
        heads_in_l = [cid for cid in sig_map if cid.startswith(f"head{l}_")]
        if heads_in_l:
            heads_in_l.sort(key=lambda cid: sig_map[cid]["p_value"])
            top_heads_per_layer.append(heads_in_l[0])
    top_heads_per_layer.sort(key=lambda cid: sig_map[cid]["p_value"])

    remaining_heads = [cid for cid in sig_map if cid.startswith("head") and cid not in top_heads_per_layer]
    remaining_heads.sort(key=lambda cid: sig_map[cid]["p_value"])

    planned = layer_cids + top_heads_per_layer + remaining_heads
    return planned[:budget]

def evaluate_candidate_intervention(env, candidate_id, n_trials=N_FLIP_TRIALS, seed=1):
    rng = np.random.default_rng(seed)
    delta_logits = []
    flips = 0

    is_head = candidate_id.startswith("head")
    if is_head:
        rest = candidate_id[len("head"):]
        layer_idx, head_idx = map(int, rest.split("_"))

    for _ in range(n_trials):
        T = int(rng.integers(0, 10))
        n_clean = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))
        n_donor = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))
        while n_donor == n_clean:
            n_donor = int(rng.integers(env.operand_range[0], env.operand_range[1] + 1))

        clean_prompt, donor_prompt = f"{T} + {n_clean} =", f"{T} + {n_donor} ="
        clean_tokens, donor_tokens = model.to_tokens(clean_prompt), model.to_tokens(donor_prompt)
        if clean_tokens.shape[-1] != donor_tokens.shape[-1]: continue

        clean_ans_tok = model.to_tokens(f" {(T + n_clean) % 10}", prepend_bos=False)[0, 0].item()
        donor_ans_tok = model.to_tokens(f" {(T + n_donor) % 10}", prepend_bos=False)[0, 0].item()

        clean_logits_orig = model(clean_tokens)
        orig_action = int(clean_logits_orig[0, -1, :].argmax().item())
        clean_margin = (clean_logits_orig[0, -1, donor_ans_tok] - clean_logits_orig[0, -1, clean_ans_tok]).item()

        _, donor_cache = model.run_with_cache(donor_tokens)

        if is_head:
            def patch_hook(act, hook, h=head_idx):
                act[:, -1, h, :] = donor_cache[hook.name][:, -1, h, :]
                return act
            hook_name = f"blocks.{layer_idx}.attn.hook_z"
            patched_logits = model.run_with_hooks(clean_tokens, fwd_hooks=[(hook_name, patch_hook)])
        else:
            l_idx = int(candidate_id[len("layer"):])
            def patch_hook(act, hook):
                act[:, -1, :] = donor_cache[hook.name][:, -1, :]
                return act
            hook_name = f"blocks.{l_idx}.hook_resid_post"
            patched_logits = model.run_with_hooks(clean_tokens, fwd_hooks=[(hook_name, patch_hook)])

        patched_action = int(patched_logits[0, -1, :].argmax().item())
        patched_margin = (patched_logits[0, -1, donor_ans_tok] - patched_logits[0, -1, clean_ans_tok]).item()

        if patched_action != orig_action: flips += 1
        delta_logits.append(patched_margin - clean_margin)
        del donor_cache

    valid_trials = len(delta_logits)
    avg_delta_logit = float(np.mean(delta_logits)) if valid_trials > 0 else 0.0
    flip_rate = flips / valid_trials if valid_trials > 0 else 0.0
    return avg_delta_logit, flip_rate

# =============================================================================
# Pipeline Execution
# =============================================================================

env = Environment(model, seed=0)
print(f"\n[collection] gathering {N_STEPS}-step trajectory for 156 candidates...")
matrix, action_onehot, n_anomalies = collect_trajectory(env, N_STEPS)
print(f"[collection] completed. Generation anomalies (fallback used): {n_anomalies}/{N_STEPS}")

print("\n[Stage 1] batch Endogeneity screening (PCA-reduced Granger Causality)...")
endog_results = test_endogeneity_batch(matrix, action_onehot, CANDIDATE_IDS)

world_model = WorldModel()
self_graph_nodes = {}
evidence_mgr = EvidenceManager(world_model, self_graph_nodes)

for cid, r in endog_results.items():
    evidence_mgr.record_endogeneity(cid, r["p_value"], r["significant"])

round_candidates = plan_round_stratified(endog_results, budget=BUDGET)
print(f"\n[Stage 2] testing top {len(round_candidates)} candidates via Stratified Allocation (budget={BUDGET}):")

for cid in round_candidates:
    delta_logit, flip_rate = evaluate_candidate_intervention(env, cid)
    handle_id = f"donor_activation@{cid}"
    evidence_mgr.record_intervention(cid, handle_id, delta_logit, flip_rate)
    kuv = KnowledgeUnitView(self_graph_nodes[cid], world_model)
    print(f"  {cid:<10}: delta_logit={delta_logit:+.3f}, flip_rate={flip_rate:.2f} -> {kuv}")

print("\n" + "=" * 60)
print("WorldModel Summary (Statistically Corrected Open Discovery Round):")
print(world_model.self_model_summary())

In [ ]:
"""
Phase 1: Cognitive State Persistence (SelfGraph Storage Engine)
Exports and imports discovered causal mechanisms, world model metadata,
and calibrated steering vectors to persistent storage.
"""

import json
import os
import numpy as np
import torch
from dataclasses import asdict

class SelfGraph:
    """
    Persistent storage and retrieval engine for the Cognitive Layer.
    Manages SelfGraphNode representations, causal metadata, and probe steering vectors.
    """
    def __init__(self, world_model=None, self_graph_nodes=None, layer_dirs=None, head_dirs=None):
        self.world_model = world_model
        self.self_graph_nodes = self_graph_nodes or {}
        self.layer_dirs = layer_dirs or {}
        self.head_dirs = head_dirs or {}

    def save(self, base_filename: str = "cognitive_layer_state"):
        """
        Saves metadata to JSON and vector arrays to compressed NPZ format.
        """
        json_path = f"{base_filename}_metadata.json"
        npz_path = f"{base_filename}_vectors.npz"

        # 1. Prepare JSON Metadata payload
        nodes_data = {}
        for node_id, sg_node in self.sg_nodes_to_dict().items():
            wm_node = self.world_model.nodes.get(node_id) if self.world_model else None
            nodes_data[node_id] = {
                "node_id": sg_node["node_id"],
                "kind": sg_node["kind"],
                "kind_confidence": sg_node["kind_confidence"],
                "status": wm_node.status.value if wm_node else "unresolved",
                "endogeneity_p_value": wm_node.endogeneity_p_value if wm_node else None,
                "delta_logit": wm_node.delta_logit if wm_node else None,
                "flip_rate": wm_node.flip_rate if wm_node else None,
                "best_handle_id": wm_node.best_handle_id if wm_node else None,
                "evidence": sg_node.get("evidence", {})
            }

        summary_data = {
            "version": "1.0_phase1",
            "accepted_mechanisms": [nid for nid, data in nodes_data.items() if data["status"] == "accepted"],
            "rejected_decoys": [nid for nid, data in nodes_data.items() if data["status"] == "rejected"],
            "nodes": nodes_data
        }

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(summary_data, f, indent=2, ensure_ascii=False)

        # 2. Prepare NPZ Tensors payload
        vector_dict = {}
        for layer_id, vec in self.layer_dirs.items():
            vector_dict[f"layer_{layer_id}"] = np.array(vec)
        for (layer_id, head_id), vec in self.head_dirs.items():
            vector_dict[f"head_{layer_id}_{head_id}"] = np.array(vec)

        np.savez_compressed(npz_path, **vector_dict)

        print("=" * 65)
        print(f"[SelfGraph Persistence] SUCCESSFULLY SAVED COGNITIVE STATE")
        print(f"  ├── Metadata JSON : '{json_path}' ({os.path.getsize(json_path)} bytes)")
        print(f"  └── Vectors NPZ   : '{npz_path}' ({os.path.getsize(npz_path)} bytes)")
        print(f"  └── Accepted Mechanisms Saved: {summary_data['accepted_mechanisms']}")
        print("=" * 65)

    def sg_nodes_to_dict(self):
        result = {}
        for nid, node in self.self_graph_nodes.items():
            result[nid] = {
                "node_id": node.node_id,
                "kind": node.kind.value if hasattr(node.kind, 'value') else str(node.kind),
                "kind_confidence": node.kind_confidence,
                "evidence": node.evidence
            }
        return result

    @classmethod
    def load(cls, base_filename: str = "cognitive_layer_state"):
        """
        Loads persistent cognitive metadata and steering vectors from disk.
        """
        json_path = f"{base_filename}_metadata.json"
        npz_path = f"{base_filename}_vectors.npz"

        if not os.path.exists(json_path) or not os.path.exists(npz_path):
            raise FileNotFoundError(f"Cannot find '{json_path}' or '{npz_path}'")

        with open(json_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)

        npz_data = np.load(npz_path)
        layer_dirs = {}
        head_dirs = {}

        for key in npz_data.files:
            if key.startswith("layer_"):
                l_idx = int(key.replace("layer_", ""))
                layer_dirs[l_idx] = npz_data[key]
            elif key.startswith("head_"):
                parts = key.replace("head_", "").split("_")
                l_idx, h_idx = int(parts[0]), int(parts[1])
                head_dirs[(l_idx, h_idx)] = npz_data[key]

        print(f"[SelfGraph Persistence] SUCCESSFULLY LOADED COGNITIVE STATE")
        print(f"  ├── Accepted Mechanisms Loaded: {metadata['accepted_mechanisms']}")
        print(f"  └── Total Vectors Restored   : {len(layer_dirs)} layers, {len(head_dirs)} heads")

        return metadata, layer_dirs, head_dirs

    @staticmethod
    def get_accepted_steering_vectors(metadata, layer_dirs, device="cuda"):
        """
        Extracts only ACCEPTED causal steering vectors ready for live injection.
        """
        accepted_ids = metadata.get("accepted_mechanisms", [])
        steering_tensors = {}

        for cid in accepted_ids:
            if cid.startswith("layer"):
                l_idx = int(cid.replace("layer", ""))
                if l_idx in layer_dirs:
                    steering_tensors[l_idx] = torch.tensor(
                        layer_dirs[l_idx], dtype=torch.float32, device=device
                    )

        return steering_tensors


# =============================================================================
# EXECUTION: Save current WorldModel & Test Reloading
# =============================================================================

# 1. Instantiating and saving current state
self_graph_storage = SelfGraph(
    world_model=world_model,
    self_graph_nodes=self_graph_nodes,
    layer_dirs=layer_dirs,
    head_dirs=head_dirs
)

self_graph_storage.save("gpt2_cognitive_layer")

# 2. Test Loading stored cognitive state independently
loaded_metadata, loaded_layer_dirs, loaded_head_dirs = SelfGraph.load("gpt2_cognitive_layer")

# 3. Extract ready-to-use Steering Tensors for live control
active_steering_vectors = SelfGraph.get_accepted_steering_vectors(
    loaded_metadata, loaded_layer_dirs, device=DEVICE
)

print(f"\n[Validation] Active Steering Tensors ready for layers: {list(active_steering_vectors.keys())}")